In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, auc,
                           f1_score, matthews_corrcoef, cohen_kappa_score,
                           accuracy_score, precision_score, recall_score)
from sklearn.preprocessing import label_binarize
import cv2
from tqdm import tqdm
import glob
import random
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

import lime
from lime import lime_image
from skimage.segmentation import mark_boundaries

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

class Config:
    DATASET_PATH = '/kaggle/input/cervical-cancer-largest-dataset-sipakmed'
    IMG_SIZE = 224
    BATCH_SIZE = 16
    FUSION_DIM = 512
    CROSS_ATTENTION_HEADS = 8
    MORPHOLOGICAL_FEATURE_DIM = 256
    TEXTURAL_FEATURE_DIM = 128
    NUCLEUS_FEATURE_DIM = 128
    CYTOPLASM_FEATURE_DIM = 128
    
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    EPOCHS = 20
    PATIENCE = 5 
    RESULTS_PATH = "enhanced_results/"
    ENSEMBLE_PATH = "ensemble_models/"

config = Config()

os.makedirs(config.RESULTS_PATH, exist_ok=True)
os.makedirs(config.ENSEMBLE_PATH, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def load_sipakmed_data():
    print("Loading SipakMed dataset...")
    class_folders = ['Dyskeratotic', 'Koilocytotic', 'Metaplastic', 'Parabasal', 'Superficial-Intermediate']
    class_names = class_folders
    X, y = [], []
    for idx, class_folder in enumerate(class_folders):
        folder_path = os.path.join(config.DATASET_PATH, f"im_{class_folder}", f"im_{class_folder}", "CROPPED")
        if not os.path.exists(folder_path):
            print(f"Folder not found: {folder_path}"); continue
        image_paths = glob.glob(os.path.join(folder_path, '*.bmp'))
        print(f"Loading {len(image_paths)} images from class: {class_folder}")
        for img_path in tqdm(image_paths, desc=class_folder):
            img = cv2.imread(img_path)
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (config.IMG_SIZE, config.IMG_SIZE))
            X.append(img)
            y.append(idx)
    X_arr = np.array(X, dtype=np.uint8)
    y_arr = np.array(y)
    print(f"Dataset loaded: {len(X_arr)} images, {len(class_names)} classes")
    return X_arr, y_arr, class_names

class SipakmedDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        image, label = self.images[idx], self.labels[idx]
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

def get_transforms():
    train_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomRotation(25), transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.ToPILImage(), transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, val_transform

class NucleusSegmentationModule(nn.Module):
    def __init__(self, in_channels=3, out_channels=128):
        super(NucleusSegmentationModule, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
    def forward(self, x):
        x = self.encoder(x)
        x = self.pool(x)
        return x.view(x.size(0), -1)

class CytoplasmSegmentationModule(nn.Module):
    def __init__(self, in_channels=3, out_channels=128):
        super(CytoplasmSegmentationModule, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, out_channels, kernel_size=5, padding=2),
            nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
    def forward(self, x):
        x = self.encoder(x)
        x = self.pool(x)
        return x.view(x.size(0), -1)

class MorphologicalFeatureExtractor(nn.Module):
    def __init__(self, feature_dim=256):
        super().__init__()
        self.conv_morph1 = nn.Conv2d(3, 64, kernel_size=3, padding='same')
        self.conv_morph2 = nn.Conv2d(3, 128, kernel_size=5, padding='same')
        self.conv_morph3 = nn.Conv2d(3, 128, kernel_size=7, padding='same')
        self.pool1 = nn.MaxPool2d(2); self.pool2 = nn.MaxPool2d(4); self.pool3 = nn.MaxPool2d(8)
        self.conv_fusion = nn.Conv2d(64 + 128 + 128, feature_dim, kernel_size=1)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.bn = nn.BatchNorm2d(feature_dim); self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        morph1, morph2, morph3 = self.pool1(self.relu(self.conv_morph1(x))), self.pool2(self.relu(self.conv_morph2(x))), self.pool3(self.relu(self.conv_morph3(x)))
        target_size = morph3.shape[2:]
        morph1_resized = F.interpolate(morph1, size=target_size, mode='bilinear', align_corners=False)
        morph2_resized = F.interpolate(morph2, size=target_size, mode='bilinear', align_corners=False)
        fused_morph = torch.cat([morph1_resized, morph2_resized, morph3], dim=1)
        morph_features = self.bn(self.relu(self.conv_fusion(fused_morph)))
        return torch.flatten(self.global_pool(morph_features), 1)

class TexturalFeatureExtractor(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.text_conv1 = nn.Conv2d(3, 32, kernel_size=3, padding='same')
        self.text_conv2 = nn.Conv2d(32, 64, kernel_size=5, padding='same')
        self.compress_conv = nn.Conv2d(64, feature_dim, kernel_size=1)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.bn = nn.BatchNorm2d(feature_dim); self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        text_features = self.bn(self.relu(self.compress_conv(self.relu(self.text_conv2(self.relu(self.text_conv1(x)))))))
        return torch.flatten(self.global_pool(text_features), 1)

class CrossModalAttentionFusion(nn.Module):
    def __init__(self, swin_dim, morph_dim, text_dim, nucleus_dim, cytoplasm_dim, fusion_dim=512, num_heads=8):
        super().__init__()
        self.swin_proj = nn.Linear(swin_dim, fusion_dim); self.morph_proj = nn.Linear(morph_dim, fusion_dim); self.text_proj = nn.Linear(text_dim, fusion_dim)
        self.nucleus_proj = nn.Linear(nucleus_dim, fusion_dim)
        self.cytoplasm_proj = nn.Linear(cytoplasm_dim, fusion_dim)
        self.cross_attention = nn.MultiheadAttention(fusion_dim, num_heads, batch_first=True)
        self.self_attention = nn.MultiheadAttention(fusion_dim, num_heads // 2, batch_first=True)
        self.ln1 = nn.LayerNorm(fusion_dim); self.ln2 = nn.LayerNorm(fusion_dim); self.ln3 = nn.LayerNorm(fusion_dim)
        self.ffn = nn.Sequential(nn.Linear(fusion_dim, fusion_dim * 2), nn.GELU(), nn.Dropout(0.1), nn.Linear(fusion_dim * 2, fusion_dim))
    def forward(self, swin_features, morph_features, text_features, nucleus_features, cytoplasm_features):
        swin_proj, morph_proj, text_proj = self.swin_proj(swin_features).unsqueeze(1), self.morph_proj(morph_features).unsqueeze(1), self.text_proj(text_features).unsqueeze(1)
        nucleus_proj = self.nucleus_proj(nucleus_features).unsqueeze(1)
        cytoplasm_proj = self.cytoplasm_proj(cytoplasm_features).unsqueeze(1)
        all_features = torch.cat([swin_proj, morph_proj, text_proj, nucleus_proj, cytoplasm_proj], dim=1)
        attn_output, _ = self.cross_attention(all_features, all_features, all_features); attn_output = self.ln1(all_features + attn_output)
        self_attn_output, _ = self.self_attention(attn_output, attn_output, attn_output); self_attn_output = self.ln2(attn_output + self_attn_output)
        ffn_output = self.ffn(self_attn_output); ffn_output = self.ln3(self_attn_output + ffn_output)
        return torch.mean(ffn_output, dim=1)

class EnhancedCytopathViT(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.swin_base = timm.create_model('swin_base_patch4_window7_224_in22k', pretrained=True, num_classes=0)
        swin_dim = self.swin_base.num_features
        for param in self.swin_base.parameters(): param.requires_grad = False
        for param in self.swin_base.layers[-2:].parameters(): param.requires_grad = True
        self.morph_extractor = MorphologicalFeatureExtractor(config.MORPHOLOGICAL_FEATURE_DIM)
        self.text_extractor = TexturalFeatureExtractor(config.TEXTURAL_FEATURE_DIM)
        self.nucleus_extractor = NucleusSegmentationModule(out_channels=config.NUCLEUS_FEATURE_DIM)
        self.cytoplasm_extractor = CytoplasmSegmentationModule(out_channels=config.CYTOPLASM_FEATURE_DIM)
        self.fusion_layer = CrossModalAttentionFusion(
            swin_dim, config.MORPHOLOGICAL_FEATURE_DIM, config.TEXTURAL_FEATURE_DIM,
            config.NUCLEUS_FEATURE_DIM, config.CYTOPLASM_FEATURE_DIM,
            config.FUSION_DIM, config.CROSS_ATTENTION_HEADS
        )
        self.fc1 = nn.Linear(config.FUSION_DIM, 512); self.bn1 = nn.BatchNorm1d(512); self.dropout1 = nn.Dropout(0.3)
        self.residual_dense = nn.Linear(config.FUSION_DIM, 512)
        self.fc2 = nn.Linear(512, 256); self.bn2 = nn.BatchNorm1d(256); self.dropout2 = nn.Dropout(0.2)
        self.output = nn.Linear(256, num_classes); self.gelu = nn.GELU()
    def forward(self, x):
        swin_features = self.swin_base(x)
        morph_features = self.morph_extractor(x)
        text_features = self.text_extractor(x)
        nucleus_features = self.nucleus_extractor(x)
        cytoplasm_features = self.cytoplasm_extractor(x)
        fused_features = self.fusion_layer(
            swin_features, morph_features, text_features, nucleus_features, cytoplasm_features
        )
        x = self.dropout1(self.bn1(self.gelu(self.fc1(fused_features))))
        residual = self.residual_dense(fused_features); x = self.gelu(x + residual)
        x = self.dropout2(self.bn2(self.gelu(self.fc2(x))))
        return self.output(x)

def calculate_comprehensive_metrics(y_true, y_pred, y_pred_proba, class_names):
    metrics = {'accuracy': accuracy_score(y_true, y_pred), 'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
               'f1_macro': f1_score(y_true, y_pred, average='macro'), 'precision_weighted': precision_score(y_true, y_pred, average='weighted'),
               'recall_weighted': recall_score(y_true, y_pred, average='weighted'), 'mcc': matthews_corrcoef(y_true, y_pred),
               'cohen_kappa': cohen_kappa_score(y_true, y_pred)}
    aucs = []
    for i in range(len(class_names)):
        binary_true = (y_true == i).astype(int)
        if len(np.unique(binary_true)) > 1:
            fpr, tpr, _ = roc_curve(binary_true, y_pred_proba[:, i]); aucs.append(auc(fpr, tpr))
    metrics['auc_overall'] = np.mean(aucs) if aucs else 0.0
    return metrics

def plot_fold_history(history, fold):
    plt.figure(figsize=(18, 5)); plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss'); plt.plot(history['val_loss'], label='Validation Loss')
    plt.title(f'Fold {fold+1} - Loss Curve'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy'); plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title(f'Fold {fold+1} - Accuracy Curve'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(config.RESULTS_PATH, f"fold_{fold+1}_curves.png")); plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names, fold, is_ensemble=False):
    title_prefix = "Ensemble" if is_ensemble else f"Fold {fold+1}"
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{title_prefix} - Confusion Matrix'); plt.ylabel('True Label'); plt.xlabel('Predicted Label'); plt.tight_layout()
    filename = "ensemble_confusion_matrix.png" if is_ensemble else f"fold_{fold+1}_confusion_matrix.png"
    plt.savefig(os.path.join(config.RESULTS_PATH, filename), dpi=300); plt.show()

def plot_roc_curves(y_true, y_pred_proba, class_names, fold, is_ensemble=False):
    title_prefix = "Ensemble" if is_ensemble else f"Fold {fold+1}"
    y_true_binarized = label_binarize(y_true, classes=range(len(class_names)))
    n_classes = len(class_names)
    fpr, tpr, roc_auc = dict(), dict(), dict()
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_binarized.ravel(), y_pred_proba.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    for i in range(n_classes): fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_proba[:, i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes): mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
    plt.figure(figsize=(12, 8))
    plt.plot(fpr["micro"], tpr["micro"], label=f'Micro-average (AUC = {roc_auc["micro"]:0.3f})', color='deeppink', linestyle=':', linewidth=4)
    plt.plot(fpr["macro"], tpr["macro"], label=f'Macro-average (AUC = {roc_auc["macro"]:0.3f})', color='navy', linestyle=':', linewidth=4)
    for i, class_name in enumerate(class_names): plt.plot(fpr[i], tpr[i], lw=2, label=f'{class_name} (AUC = {auc(fpr[i], tpr[i]):0.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2); plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title(f'{title_prefix} - ROC Curves')
    plt.legend(loc="lower right"); plt.grid(True, alpha=0.3); plt.tight_layout()
    filename = "ensemble_roc_curves.png" if is_ensemble else f"fold_{fold+1}_roc_curves.png"
    plt.savefig(os.path.join(config.RESULTS_PATH, filename), dpi=300); plt.show()

def explain_with_lime(model, X_sample_raw, y_sample_raw, class_names, num_samples=3, save_path=None):
    model.eval(); _, val_transform = get_transforms()
    def predict_fn(images):
        model.eval()
        processed_images = [val_transform((img * 255).astype(np.uint8)) for img in images]
        batch = torch.stack(processed_images).to(device)
        with torch.no_grad(): outputs = model(batch); probs = F.softmax(outputs, dim=1)
        return probs.cpu().numpy()
    explainer = lime_image.LimeImageExplainer(); plt.figure(figsize=(8, 4 * num_samples))
    for i in range(min(num_samples, len(X_sample_raw))):
        image_raw = X_sample_raw[i]
        explanation = explainer.explain_instance(image_raw.astype(np.double) / 255.0, predict_fn, top_labels=len(class_names), hide_color=0, num_samples=1000)
        temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], positive_only=False, num_features=10, hide_rest=False)
        overlay = image_raw.copy(); positive_mask = np.where(mask > 0, 1, 0); negative_mask = np.where(mask < 0, 1, 0)
        overlay[positive_mask == 1] = (0, 255, 0); overlay[negative_mask == 1] = (255, 0, 0)
        blended_image = cv2.addWeighted(image_raw, 0.7, overlay, 0.3, 0)
        plt.subplot(num_samples, 2, i * 2 + 1); plt.imshow(image_raw); plt.title(f'Original: {class_names[y_sample_raw[i]]}'); plt.axis('off')
        plt.subplot(num_samples, 2, i * 2 + 2); plt.imshow(blended_image); plt.title('LIME Overlay (Green+/Red-)'); plt.axis('off')
    plt.tight_layout();
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def explain_ensemble_with_lime(fold_models, X_sample_raw, y_sample_raw, class_names, num_samples=3, save_path=None):
    _, val_transform = get_transforms()
    def ensemble_predict_fn(images):
        all_probs = []
        for model in fold_models:
            model.eval()
            processed_images = [val_transform((img * 255).astype(np.uint8)) for img in images]
            batch = torch.stack(processed_images).to(device)
            with torch.no_grad(): outputs = model(batch); probs = F.softmax(outputs, dim=1)
            all_probs.append(probs.cpu().numpy())
        return np.mean(all_probs, axis=0)
    explainer = lime_image.LimeImageExplainer(); plt.figure(figsize=(8, 4 * num_samples))
    for i in range(min(num_samples, len(X_sample_raw))):
        image_raw = X_sample_raw[i]
        explanation = explainer.explain_instance(image_raw.astype(np.double) / 255.0, ensemble_predict_fn, top_labels=len(class_names), hide_color=0, num_samples=1000)
        temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], positive_only=False, num_features=10, hide_rest=False)
        overlay = image_raw.copy(); positive_mask = np.where(mask > 0, 1, 0); negative_mask = np.where(mask < 0, 1, 0)
        overlay[positive_mask == 1] = (0, 255, 0); overlay[negative_mask == 1] = (255, 0, 0)
        blended_image = cv2.addWeighted(image_raw, 0.7, overlay, 0.3, 0)
        plt.subplot(num_samples, 2, i * 2 + 1); plt.imshow(image_raw); plt.title(f'Original: {class_names[y_sample_raw[i]]}'); plt.axis('off')
        plt.subplot(num_samples, 2, i * 2 + 2); plt.imshow(blended_image); plt.title('Ensemble LIME Overlay'); plt.axis('off')
    plt.tight_layout();
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
def enhanced_k_fold_cross_validation(X, y, class_names, n_splits=5):
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    
    fold_results, fold_models = [], []
    train_transform, val_transform = get_transforms()

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_val, y_train_val)):
        print(f"\n{'='*20} Fold {fold+1}/{n_splits} {'='*20}")
        X_train_fold, X_val_fold = X_train_val[train_idx], X_train_val[val_idx]
        y_train_fold, y_val_fold = y_train_val[train_idx], y_train_val[val_idx]

        train_dataset = SipakmedDataset(X_train_fold, y_train_fold, transform=train_transform)
        val_dataset = SipakmedDataset(X_val_fold, y_val_fold, transform=val_transform)
        train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

        model = EnhancedCytopathViT(num_classes=len(class_names)).to(device)
        if fold == 0: print(f"Model Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.3, patience=7)

        best_val_acc = 0; epochs_no_improve = 0
        history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
        
        for epoch in range(config.EPOCHS):
            model.train()
            train_loss, train_all_preds, train_all_labels = 0.0, [], []
            for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.EPOCHS} [Train]"):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad(); outputs = model(images); loss = criterion(outputs, labels); loss.backward(); optimizer.step()
                train_loss += loss.item()
                train_all_preds.extend(torch.max(outputs, 1)[1].cpu().numpy()); train_all_labels.extend(labels.cpu().numpy())

            model.eval()
            val_loss, val_all_preds, val_all_labels = 0.0, [], []
            with torch.no_grad():
                for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{config.EPOCHS} [Val]"):
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images); loss = criterion(outputs, labels); val_loss += loss.item()
                    val_all_preds.extend(torch.max(outputs, 1)[1].cpu().numpy()); val_all_labels.extend(labels.cpu().numpy())
            
            train_acc = accuracy_score(train_all_labels, train_all_preds); val_acc = accuracy_score(val_all_labels, val_all_preds)
            avg_train_loss = train_loss / len(train_loader); avg_val_loss = val_loss / len(val_loader)
            history['train_loss'].append(avg_train_loss); history['train_acc'].append(train_acc); history['val_loss'].append(avg_val_loss); history['val_acc'].append(val_acc)
            scheduler.step(avg_val_loss)

            print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc; torch.save(model.state_dict(), f"{config.ENSEMBLE_PATH}/model_fold_{fold+1}.pth"); epochs_no_improve = 0
            else: epochs_no_improve += 1
            if epochs_no_improve >= config.PATIENCE: print(f"Early stopping triggered after {epoch+1} epochs."); break
        
        plot_fold_history(history, fold)
        
        best_model = EnhancedCytopathViT(num_classes=len(class_names)).to(device)
        best_model.load_state_dict(torch.load(f"{config.ENSEMBLE_PATH}/model_fold_{fold+1}.pth"))
        fold_models.append(best_model)

        best_model.eval()
        all_pred_proba, all_preds, all_labels = [], [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device); outputs = best_model(images)
                all_pred_proba.extend(F.softmax(outputs, dim=1).cpu().numpy())
                all_preds.extend(torch.max(outputs, 1)[1].cpu().numpy()); all_labels.extend(labels.numpy())
        
        y_true, y_pred, y_proba = np.array(all_labels), np.array(all_preds), np.array(all_pred_proba)
        fold_metrics = calculate_comprehensive_metrics(y_true, y_pred, y_proba, class_names)
        fold_results.append(fold_metrics)
        print(f"\nFold {fold+1} Detailed Report:\n", classification_report(y_true, y_pred, target_names=class_names, digits=6))
        print(f"Fold {fold+1} MCC: {fold_metrics['mcc']:.6f}, Cohen's Kappa: {fold_metrics['cohen_kappa']:.6f}")
        plot_confusion_matrix(y_true, y_pred, class_names, fold)
        plot_roc_curves(y_true, y_proba, class_names, fold)
        
        sample_indices = np.random.choice(len(X_val_fold), min(30, len(X_val_fold)), replace=False)
        explain_with_lime(best_model, X_val_fold[sample_indices], y_val_fold[sample_indices], class_names, num_samples=30, save_path=f"{config.RESULTS_PATH}/lime_fold_{fold+1}.png")
    print(f"\n{'='*50}\nENSEMBLE MODEL EVALUATION\n{'='*50}")
    test_dataset = SipakmedDataset(X_test, y_test, transform=val_transform)
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
    ensemble_predictions = np.zeros((len(y_test), len(class_names)))
    for model in fold_models:
        model.eval(); fold_preds = []
        with torch.no_grad():
            for images, _ in test_loader:
                images = images.to(device); outputs = model(images)
                fold_preds.extend(F.softmax(outputs, dim=1).cpu().numpy())
        ensemble_predictions += np.array(fold_preds)
    ensemble_predictions /= len(fold_models)
    ensemble_pred_classes = np.argmax(ensemble_predictions, axis=1)
    
    ensemble_metrics = calculate_comprehensive_metrics(y_test, ensemble_pred_classes, ensemble_predictions, class_names)
    print("\nEnsemble Test Results:")
    for key, value in ensemble_metrics.items(): print(f"{key.replace('_', ' ').title()}: {value:.6f}")
    print("\nEnsemble Classification Report:\n", classification_report(y_test, ensemble_pred_classes, target_names=class_names, digits=6))
    plot_confusion_matrix(y_test, ensemble_pred_classes, class_names, fold=-1, is_ensemble=True)
    plot_roc_curves(y_test, ensemble_predictions, class_names, fold=-1, is_ensemble=True)
    
    sample_indices = np.random.choice(len(X_test), min(30, len(X_test)), replace=False)
    explain_ensemble_with_lime(fold_models, X_test[sample_indices], y_test[sample_indices], class_names, num_samples=30, save_path=f"{config.RESULTS_PATH}/lime_ensemble.png")

    metrics_df = pd.DataFrame(fold_results); means = metrics_df.mean(); stds = metrics_df.std()
    results_summary = {'fold_results': fold_results, 'cv_means': means.to_dict(), 'cv_stds': stds.to_dict(), 'ensemble_metrics': ensemble_metrics, 'class_names': class_names}
    with open(f"{config.RESULTS_PATH}/comprehensive_results.pkl", 'wb') as f: pickle.dump(results_summary, f)
    return fold_models, ensemble_metrics, results_summary

def main():
    start_time = time.time()
    print("Enhanced CervicalPathViT with Swin Transformer (PyTorch Version)"); print("=" * 60)
    X, y, class_names = load_sipakmed_data()
    fold_models, ensemble_metrics, results_summary = enhanced_k_fold_cross_validation(X, y, class_names)
    
    n_splits, means, stds = len(results_summary['fold_results']), results_summary['cv_means'], results_summary['cv_stds']
    results_df = pd.DataFrame({
        'Fold': [f'Fold {i+1}' for i in range(n_splits)] + ['Mean', 'Std', 'Ensemble'],
        'Accuracy': [res['accuracy'] for res in results_summary['fold_results']] + [means['accuracy'], stds['accuracy'], ensemble_metrics['accuracy']],
        'F1-Weighted': [res['f1_weighted'] for res in results_summary['fold_results']] + [means['f1_weighted'], stds['f1_weighted'], ensemble_metrics['f1_weighted']],
        'MCC': [res['mcc'] for res in results_summary['fold_results']] + [means['mcc'], stds['mcc'], ensemble_metrics['mcc']],
        'Cohen Kappa': [res['cohen_kappa'] for res in results_summary['fold_results']] + [means['cohen_kappa'], stds['cohen_kappa'], ensemble_metrics['cohen_kappa']],
        'AUC': [res['auc_overall'] for res in results_summary['fold_results']] + [means['auc_overall'], stds['auc_overall'], ensemble_metrics['auc_overall']]
    })
    
    print(f"\n{'='*60}\nPUBLICATION-READY RESULTS TABLE\n{'='*60}"); print(results_df.round(6))
    
    end_time = time.time(); execution_time = end_time - start_time
    hours, rem = divmod(execution_time, 3600); minutes, seconds = divmod(rem, 60)
    print(f"\nTotal execution time: {int(hours)}h {int(minutes)}m {seconds:.2f}s")
    
    return fold_models, ensemble_metrics, results_summary

if __name__ == "__main__":
    fold_models, ensemble_metrics, results_summary = main()